# Verification

A first answer is a draft. Verification checks it before we use it.

In this notebook, we will:

- check a decision with shop rules;
- ask a model to judge open-ended work;
- fail safely when the judge returns bad data;
- revise a draft from feedback; and
- choose when verification is worth its cost.

In [ ]:
import json
import os
import time
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.getenv("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

We will use one opened-item ticket. The shop rules apply a restocking fee, so the exact refund amount matters.

In [ ]:
from shoplab import world
from shoplab.llm import LEDGER, llm, parse_json_loose

tickets = world.load_tickets()
orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}

ticket = next(
    item
    for rows in tickets.values()
    for item in rows
    if item["ticket_id"] == "TKT-2205"
)
order = orders[ticket["order_id"]]
customer = customers[ticket["customer_id"]]

## 1. Use a deterministic verifier when possible

If code can calculate the correct answer, use code. It is fast, cheap, and repeatable.

In [ ]:
ticket_text = json.dumps({key: value for key, value in ticket.items() if key != "gold"})
draft_text = llm(
    ticket_text + "\nDecide from this text alone. Return JSON with decision, policy_id, and refund_usd.",
    model=MODEL,
)
draft = parse_json_loose(draft_text)

print("draft:", draft)

In [ ]:
from shoplab.verify import check_decision

check = check_decision(draft, ticket, order, customer)

print("agrees:", check["agrees"])
print("wrong fields:", check["mismatch"])
print("correct answer:", check["gold"])

## 2. Use a model judge for open-ended work

Code can compare exact fields. It is harder to grade an explanation. For that, we can give a stronger model a clear rubric.

In [ ]:
JUDGE_SYSTEM = (
    "Grade the answer using the rubric. Return only JSON with "
    "pass (true or false), score (0 to 1), and reasons."
)

def small_judge(question, answer, rubric):
    prompt = f"""Question:
{question}

Answer:
{answer}

Rubric:
{rubric}"""

    reply = llm(prompt, system=JUDGE_SYSTEM, model=STRONG_MODEL)

    try:
        verdict = parse_json_loose(reply)
        return {
            "pass": bool(verdict.get("pass")),
            "score": float(verdict.get("score", 0)),
            "reasons": str(verdict.get("reasons", "")),
        }
    except (ValueError, TypeError, AttributeError) as error:
        return {"pass": False, "score": 0.0, "reasons": f"unusable judge reply: {error}"}

A judge is also a model. Its output can fail. Our safe default is a failing verdict, not a passing one.

In [ ]:
question = "Does this explanation support the refund?"
rubric = (
    "Pass only if the explanation names pol-restocking and shows that "
    "a 10 percent fee was removed from the item price."
)

good = "pol-restocking applies. Ten percent is removed, so $189.99 becomes $170.99."
weak = "The customer seems honest, so give a full refund."

print("good:", small_judge(question, good, rubric))
print("weak:", small_judge(question, weak, rubric))

The course provides the finished judge as shoplab.verify.judge.

In [ ]:
from shoplab.verify import judge

judge(question, good, rubric)

## 3. Draft, critique, revise

A refinement loop follows four steps:

1. Make a draft.
2. Check it.
3. Give the problem back to the drafter.
4. Stop when it passes or reaches the round limit.

In [ ]:
def refine(draft_function, check_function, max_rounds=3):
    feedback = None
    history = []

    for round_number in range(1, max_rounds + 1):
        draft = draft_function(feedback)
        feedback = check_function(draft)
        history.append(feedback)

        if feedback["pass"]:
            return {"result": draft, "rounds": round_number, "history": history}

    return {"result": draft, "rounds": max_rounds, "history": history}

In [ ]:
def write_explanation(feedback):
    prompt = "Explain in one sentence why this opened item gets a $170.99 partial refund."
    if feedback:
        prompt += " The last draft failed because: " + feedback["reasons"]
    return llm(prompt, model=MODEL)

def check_explanation(text):
    return judge(question, text, rubric)

result = refine(write_explanation, check_explanation, max_rounds=3)

print("rounds:", result["rounds"])
for number, verdict in enumerate(result["history"], 1):
    print("round", number, verdict)
print("final:", result["result"])

The round limit matters. A critic can keep finding problems forever. A production loop must have a clear end.

## 4. Verify where the risk is

A rule check uses no model. A model judge adds time and cost. Use the cheapest reliable check first, and save model judging for work that code cannot grade.

In [ ]:
started = time.perf_counter()
for _ in range(1000):
    check_decision(ticket["gold"], ticket, order, customer)
rule_time = time.perf_counter() - started

judge_start = len(LEDGER)
started = time.perf_counter()
judge(question, good, rubric)
judge_time = time.perf_counter() - started

print("1000 rule checks:", round(rule_time, 4), "seconds")
print("1 model judge:", round(judge_time, 4), "seconds")
print("model calls added:", len(LEDGER) - judge_start)

## What to remember

1. Treat the first answer as a draft.
2. Use deterministic checks when code knows the answer.
3. Give a model judge a clear rubric.
4. A broken judge reply must fail safely.
5. Bound every draft-and-revise loop.
6. Spend verification cost where mistakes matter most.

## Small practice steps

1. Change one field in ticket["gold"] and run check_decision.
2. Make the rubric require a policy id and a dollar amount.
3. Set max_rounds to one and compare the final answer.